In [1]:
#Import the required Libraries
import os
import numpy as np
from openai import OpenAI
import faiss
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
import json
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
#Load open Ai key
load_dotenv(dotenv_path=r'C:\Users\User\Agentic AI\.env')
open_api_key = os.getenv("OPEN_API_KEY")
openai_client = OpenAI(api_key=open_api_key,
                       project="proj_AiRCDdpGwAmt6xebs4kjwD8f")

In [3]:
#Load the meeting transcript
def load_meeting_text(filename):
    with open(filename,'r',encoding="utf-8") as file:
        meeting_text = file.read()

    print("Data loaded successfully!")
    return meeting_text

In [4]:
#Call the function tomload the data
meeting_text = load_meeting_text('meeting_transcript.txt')

Data loaded successfully!


In [5]:
def add_line_numbers(text):

    lines = text.splitlines()

    numbered_lines = []

    for i, line in enumerate(lines, start=1):

        if line.strip():

            numbered_lines.append(
                f"[Line {i}] {line.strip()}"
            )

    return "\n".join(numbered_lines)


numbered_transcript = add_line_numbers(
    meeting_text
)

In [6]:
#Function to chunk the text usint NLTK punkt, here we are creating chucks of 500 characters
def chunk_by_sent(text, max_chars=500):
    sentences = nltk.sent_tokenize(text)
    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) < max_chars:
            current += " " + sentence
        else:
            chunks.append(current.strip())
            current = sentence
    if current:
        chunks.append(current.strip())
    print("Number of chunks:", len(chunks))
    return chunks

In [7]:
#Chunk text nefore proceeding to Embedding in order to give better context to RAG
chunks = chunk_by_sent(numbered_transcript)
print(chunks[:3])

Number of chunks: 10
["[Line 1] MEETING TRANSCRIPT — PRODUCT DEVELOPMENT TEAM\n[Line 3] [Line 1] Sarah: Good morning everyone. Let's start our project status meeting. [Line 5] [Line 2] Sarah: Today I want us to review the remaining development work, discuss the project proposal, and identify the next action items. [Line 7] [Line 3] David: The database work is progressing well. [Line 9] [Line 4] David: I have completed the progress-tracking tables.", "[Line 11] [Line 5] David: I still need to finish the remaining database changes. [Line 13] [Line 6] Sarah: Great. Please make sure the database changes are tracked as an action item. [Line 15] [Line 7] David: Yes, I'll take care of those database changes. [Line 17] [Line 8] Priya: I've also been testing the content-generation and question-generation prompts. [Line 19] [Line 9] Priya: The results are improving, but I need to run a few more tests.", "[Line 21] [Line 10] Priya: I'll continue testing the prompts. [Line 23] [Line 11] Sarah: Goo

In [8]:
#Function to create OpenAI embeddings
def create_embeddings(chunks):
    response = openai_client.embeddings.create(
        model = 'text-embedding-3-small',
        input =  chunks   
    )

    embeddings = [item.embedding for item in response.data]

    return np.array(embeddings).astype("float32")
  

In [9]:
#Call the embeddings function to  create the embeddings
embeddings = create_embeddings(chunks)

In [10]:
#Create FAISS vector store

# dimension of embedding
d = embeddings.shape[1]
# create index
index = faiss.IndexFlatL2(d)

index.add(embeddings)
print(f'Vectors stoored into FIASS index successfull!. Total vectors={index.ntotal}')

Vectors stoored into FIASS index successfull!. Total vectors=10


In [11]:
#Create the retriever
def context_retriever(question, k=3):
    response = openai_client.embeddings.create(
      model = "text-embedding-3-small",
      input = question
    )
    question_embeddings = np.array([response.data[0].embedding]).astype("float32")
    
    distance, indices = index.search(question_embeddings,k )

    retrieved_chunks = []

    for i in indices[0]:
        retrieved_chunks.append(chunks[i])

    return retrieved_chunks
        

In [12]:
# Define the LLM
llm = LLM(
    model="gpt-5-mini",
    api_key=open_api_key
)

As this involves humans review am creating 2 Crews.

Crew-1:
Meeting Agent,
Decision Agent,
Action Agent,

Crew-2:
Review Agent,
Summary Agent

In [13]:
#CREW-1
#Create the Agents
#Agent_1 = meeting_agent
meeting_agent = Agent(
    role = "Meeting Agent",
    goal = "Analyze the meeting notes/information and identify the main points of discussion",
    backstory = """You are an experienced meeting analyst. Who carefully examine the meeting information
    and identify the important topics and discussions.Please use only the information from the context""",
    llm = llm,
    verbose = True
)

#Agent_2 = Decision Agent
decision_agent = Agent(
    role = "Decision Extractor",
    goal = "Identify the decisions made during the meeting",
    backstory = """You are an expert in extracting confirmed decisions from the business meetings.
    Distinguish between discussions, suggestions, decisions. Please don't create new information""",
    llm = llm,
    verbose = True
)

#Agent_3 = Action Agent
action_agent = Agent(
    role = "Action Item Manager",
    goal = "Identify the action items, responsible people and deadlines in the transcipt",
    backstory = """You an expertise in Project Coordination.
    Please Extarct the following:
    task
    owner
    deadline
    evidence
    transcipt line reference

    If ower is not specified say 'owner - UnResolved'
    
    If a deadline is missing please say 'Deadline is not specified'. 
    Never give random deadline or never guess a deadline.

    Always return valid JSON
    """,
    llm = llm,
    verbose = True
)



In [14]:
#Create tasks for the agents
#1. Task for analysis

analysis_task = Task(

    description = """ Analyse the retrieved meeting information. 
    Identify the relevant discussion points needed to answer the user question.

    For every important discussion point provide the following:
        Discussion point.
        Eidence.
        Transcipt Line Reference
    
    User Question: {question}
    Retrieved Meeting Information: {context}

    Please use only the information from the meeting context. Strictly avoid information outside 
    the retrieved context.
    """,

    expected_output = """
    A concise list of relevant discussion points.
    """,
    agent = meeting_agent
    )

#2. Task for decision

decision_task = Task(
    description=""" Based on the information retrieved indentify any decision relevant to the 
    user question.

    For every decision provide the following:
        Discussion point.
        Eidence.
        Transcipt Line Reference.

    If no relevant information if found, then say: 'No relevant decisions found!'
    
    User Question: {question}
    Retrieved information: {context}
    

    Only report the decisions supported by the context.
    """,    

    expected_output=""" A list of relevant decisions with supporting evidences and transcipt line references.
      Say so, if there are no relevant decisions""",
    agent = decision_agent,

    context=[
        analysis_task
    ]
)

#3. Task for Action

action_task = Task(
    description = """ Exmaine the retrived information and identiy the relevant action items to the users question.
    
    User Question: {question}
    Retrieved Information:{context}
    
    An action item can exist even when the transcript does not
    identify an owner.

    An action item can also exist even when the transcript does
    not identify a deadline.
    
    For each action item you need to identify:

    Owner
    Task
    Deadline
    Evidence
    Line reference

    If a deadline is not present, write:
    'Deadline not specified'.

    If a deadline is not present, write:
    'oNWER': 'UNRESOLVED'.

    Never guess.

    Return only valid JSON.

    Use this structure:

    {{
        "action_items": [
            {{
                "task": "...",
                "owner": "...",
                "deadline": "...",
                "evidence": "...",
                "line_reference": "..."
            }}
        ]
    }}

    """,

    expected_output=""" Valid JSON containing the extracted action items""",
    agent =  action_agent,
    context=[
        analysis_task,
        decision_task
    ]
)


In [15]:
#Create crew_1
crew_1 = Crew(
    agents = [
        meeting_agent,
        decision_agent,
        action_agent
    ],
    tasks = [
        analysis_task,
        decision_task,
        action_task
    ],
    process = Process.sequential,
    verbose = True
)

In [16]:
#CREW-2
#Create Review & Summary Agent

#Agent_4 = Review Agent
review_agent = Agent(
    role = "Meeting Report Reviewer",
    goal = """PReview the extarcted meeting information for the following:
    Accuracy
    Completeness
    Unsupported Claims
    Meeting Evidence""",
    backstory = """ You are an experienced meeting reviewer.
    You're job is to verify the following:
     Discussion points are supported.
     Decisions are supported.
     Action items are supported.
     Owners are not guessed/invented.
     Deadlines are not guessed/invented.
     Transcipt line reference numbers are included.
     User clarifications need to be incorporated.
     
     Correct unsupported information. Never guess/invent information""",

    llm = llm,
    verbose = True
)

#Agent_5 = Summary Agent
summary_agent = Agent(
    role = "Meeting Summary Specialist",
    goal = """Produce a clear MeetMind AI report using the reviewed information""",
    backstory = """ You are an intelligent AI meeting assistant.
    You're job is produce clear final report by using the review agent corrections.
    Don't add the information rejected by the reviewer.
    Clearly identify the unresolved information.
    Always include evidences and transcipt line numbers.""",
    llm = llm,
    verbose = True
)

In [17]:
#Crew -2 tasks:
#4. Task for reviewing 
review_task = Task(
    description= """ Review the extracted meeting information.
    
    User Question: {question}
    Retrieved Information: {context}

    Meeting Analysis: {analysis}
    Decisions: {decisions}
    Action Items: {action_items}
    User Clarifications: {clarifications}

    IMPORTANT:

    The USER CLARIFICATIONS represent information provided
    directly by the user after the extraction stage.

    When a user clarification provides an owner or deadline,
    update the corresponding action item with that information.

    Do NOT overwrite a user-provided owner or deadline with
    UNRESOLVED.

    Only keep UNRESOLVED when:
    1. the transcript does not provide the information, AND
    2. the user also did not provide the information.

    Check the extracted information.
    Verify the following:
     - Discussion points are supported by the transcript.
     - Decisions are supported.
     - Action items are supported.
     - Decisions have required evidence.
     - Decisions have transcript references.
     - Action items have evidence and transcript reference.
     - Owners are not guessed/invented.
     - Deadlines are not guessed/invented.
     - Always the information which cannot be verified remains UNRESOLVED.

     Correct any supported information. Never guess/invent information.
    """,

    expected_output= """
        Reviewed and corrected version of:
         - Discussion points
         - Decision points
         - Action items
         - Evidence
         - Transcript refernce
         - User clarifications
         - Unresolved information
        """,

        agent= review_agent
)

summarization_task = Task(
    description="""Create the final MeetMind AI meeting report.

    User Question:{question}
    User Clarifications: {clarifications}
    
    use ONLY the reviewed information. DO NOT include the information rejected by the Review Agent.
    Make sure to mark the unresolved information. 
    Include evidences and transcript refererences for decisions and action items.

    """,
    expected_output= """A clear final report contianing the following:
    1. Answers to the user questions.
    2. Relevant discussion points.
    3. Decisions
        Includes:
        - Decision
        - Evidences
        - Transcript References
    4. Action Items:
        Includes:
        - Task
        - Owner
        - Deadline
        - Evidence
        - Transcript References
    5. Unresolved information
    6. Executive Summary.
        """,
    agent = summary_agent,
    context = [review_task]

)

In [18]:
#Now lets create CREW-2
crew_2 = Crew(
    agents = [review_agent,
              summary_agent],
    tasks = [review_task,
             summarization_task],
    process = Process.sequential,
    verbose = True
)

In [19]:
#Function to extract JSON
def parse_action_items(action_output):
    text = str(action_output).strip()
    if text.startswith("'''json"):
        text = text[len("'''json")].strip()
    elif text.startswith("```"):
        text = text[len("```"):].strip()

    if text.endswith("```"):
        text = text[:-3].strip()
    try:
        data = json.loads(text)
        return data.get("action_items",[])
    except json.JSONDecodeError as e:
        print("WARNING: Action Agent didn't return VALID JSON!")
        print("JSON error:", e)
        print("Raw Action Agent Output:")
        print(text)

    return []
    

In [20]:
#Function to ask user for clarifications
def clarify_items(action_items):
    clarifications = []
    for item in action_items:
        task = item.get("task", "Unknown task")

        owner = item.get("owner", "UNRESOLVED")
        if ( not owner or owner.upper()=="UNRESOLVED"):
            answer = input(f"\n MeetMindAI needs some clarifications.\n"
                           f"Who is responsible for:\n"
                           f"{task}\n"
                           f"Enter the name or type 'unresolved':").strip()
            if answer.lower() == "unresolved":
                item["owner"] = "UNRESOLVED"
            else:
                item["ownner"] = answer

            clarifications.append(f"Owner clarification for {task}: {answer}")

        deadline = item.get("deadline","UNRESOLVED")
        if (not deadline or deadline.upper()=="UNRESOLVED"):
            answer = input(f"\n MeetMindAI needs some clarifications.\n"
                           f"What is the deadline for:\n"
                           f"{task}\n"
                           f"Enter a date or type 'unresolved':").strip()
            if answer.lower() == "unresolved":
                item["deadline"] = "UNRESOLVED"
            else:
                item["deadline"] = answer

            clarifications.append(f"Deadline clarification for {task}: {answer}")

    return(action_items, clarifications)

In [21]:
#Function to get the task outputs from the crews starting with crew-1
def get_crew1_outputs(result):
    analysis_output = ""
    decision_output = ""
    action_output = ""

    try:
        outputs = result.tasks_output
        for i, output in enumerate(outputs):
            print(f"\n--- Task {i + 1} ---")
            print("Output type:", type(output))
            print("Agent:", getattr(output, "agent", "Unknown"))
            print("Raw output:")
            print(getattr(output, "raw", ""))

        print("No of output tasks:",len(outputs))

        if len(outputs) >= 1:
            analysis_output = str(outputs[0].raw)

        if len(outputs) >= 2:
            decision_output = str(outputs[1].raw)

        if len(outputs) >= 3:
            action_output = str(outputs[2].raw)
    except Exception as e:
        print("ERROR: Could not read task outputs!",e)

    return(analysis_output, decision_output, action_output)

In [22]:
#Main application
async def run_meetmind():
    while True:
        #User Question
        question = input("\n Ask MeetMind AI about the meeting"
                         "(type 'exit' to Quit):").strip()

        if question.lower() == "exit":
            break
        #Retrive the data using RAG
        retrieved_chunks = context_retriever(question)
        context = "\n\n".join(retrieved_chunks)

        #Start CREW-1
        print("\n"+ "-" *25)
        print("Crew-1 Started Successfully")
        print("\n"+ "-" *25)
        extraction_result = await crew_1.kickoff_async(
            inputs = {
                "question" : question,
                "context"  : context
            }
        )
        #Capture the crew-1 outputs:
        print("\n>>> STEP 2: Crew 1 finished")
        print("Result type:", type(extraction_result))

        print("\n>>> STEP 3: Getting Crew 1 outputs")
        (analysis_output, decision_output, action_output) = get_crew1_outputs(extraction_result)
        print("\n>>> STEP 4: Crew 1 outputs captured")

        print("\nAnalysis:")
        print(analysis_output)

        print("\nDecisions:")
        print(decision_output)

        print("\nAction output:")
        print(action_output)

        print("\n>>> STEP 5: Parsing action items")
        #parse action items
        action_items = parse_action_items(action_output)
        print("\n" + "=" * 60)
        print("DEBUG - ACTION ITEMS")
        print("=" * 60)

        print("\nRaw Action Agent Output:")
        print(action_output)

        print("\nParsed Action Items:")
        print(action_items)

        print("\nNumber of Action Items:")
        print(len(action_items))

        #Human clarifications
        clarifications = []
        if action_items:
            print("\nChecking action items for missing information...")
            action_items,clarifications = clarify_items(action_items)
        #Above retrieved action items need to be concerted to JSOn
        action_items_for_review = json.dumps(
            {
                "action_items" : action_items
            },
            indent = 2
        )
        #Prepare clarifications

        if clarifications:
            clarification_text = "\n".join(clarifications)
        else:
            clarification_text = ("No additional user clarifications required")

        #Start Crew-2 to review and summarize
        print("\n"+ "-" * 25)
        print("Crew-2 Started Successfully")
        print("\n"+"-" * 25)

        review_result = await crew_2.kickoff_async(
            inputs = {
                "question" : question,
                "context" : context,
                "analysis" : analysis_output,
                "decisions" : decision_output,
                "action_items" : action_items_for_review,
                "clarifications" : clarification_text,
                "reviewed_information" : ""
            })

        #FInal Report Generation
        print("\n"+ "=" * 40)
        print("MeetMind AI Final Report")
        print("\n" + "-"* 40)
        print(review_result)
        print("\n" + "-"* 40)
        
        

In [23]:

#Start the MeetMindAI
await run_meetmind()


-------------------------
Crew-1 Started Successfully

-------------------------


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.15                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0609b5f1-84f8-449a-82ab-a7efeed4dc9d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:  Analyse the retrieved meeting information.                                                              │
│      Identify the relevant discussion points needed to answer the user question.                                │
│                                                                                                                 │
│      For every important discussion point provide the following:                                                │
│          Discussion point.                                                                                      │
│          Eidence.                                                                                               │
│          Transcipt Line Reference                                                                               │
│                                                                                                                 │
│      User Question: What are the action items that still need an owner or deadline?                             │
│      Retrieved Meeting Information: [Line 67] [Line 33] Sarah: Someone needs to estimate the remaining          │
│  development work. [Line 69] [Line 34] David: We haven't assigned that task to anyone either. [Line 71] [Line   │
│  35] Priya: And we haven't set a deadline for the estimate. [Line 73] [Line 36] Michael: We can discuss the     │
│  assignment after this meeting. [Line 75] [Line 37] Sarah: Okay. Record both the owner and deadline as          │
│  unresolved for that task. [Line 77] [Line 38] Sarah: Let's summarize the action items.                         │
│                                                                                                                 │
│  Let's leave the owner unresolved for now and come back to it after we estimate the remaining work. [Line 57]   │
│  [Line 28] Sarah: The proposal also needs to be completed before the next project review. [Line 59] [Line 29]   │
│  Sarah: We haven't decided the exact date for the review yet. [Line 61] [Line 30] David: So the deadline for    │
│  the proposal is not known yet. [Line 63] [Line 31] Sarah: Correct. We'll determine the date later. [Line 65]   │
│  [Line 32] Sarah: There is another task we need to capture.                                                     │
│                                                                                                                 │
│  [Line 93] [Line 46] Sarah: The deadline for the project proposal also needs to be determined. [Line 95] [Line  │
│  47] Sarah: The remaining development work also needs to be estimated. [Line 97] [Line 48] Sarah: The person    │
│  responsible for the estimate has not been assigned. [Line 99] [Line 49] Sarah: The deadline for completing     │
│  the estimate has also not been assigned. [Line 101] [Line 50] Sarah: These unresolved items should be          │
│  clarified before we finalize the project plan.                                                                 │
│                                                                                                                 │
│      Please use only the information from the meeting context. Strictly avoid information outside               │
│      the retrieved context.                                                                                     │
│                                                                                                                 │
│  ID: fa970820-e1a2-4798-bb32-30783c0e7f9a                                                                       │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meeting Agent                                                                                           │
│                                                                                                                 │
│  Task:  Analyse the retrieved meeting information.                                                              │
│      Identify the relevant discussion points needed to answer the user question.                                │
│                                                                                                                 │
│      For every important discussion point provide the following:                                                │
│          Discussion point.                                                                                      │
│          Eidence.                                                                                               │
│          Transcipt Line Reference                                                                               │
│                                                                                                                 │
│      User Question: What are the action items that still need an owner or deadline?                             │
│      Retrieved Meeting Information: [Line 67] [Line 33] Sarah: Someone needs to estimate the remaining          │
│  development work. [Line 69] [Line 34] David: We haven't assigned that task to anyone either. [Line 71] [Line   │
│  35] Priya: And we haven't set a deadline for the estimate. [Line 73] [Line 36] Michael: We can discuss the     │
│  assignment after this meeting. [Line 75] [Line 37] Sarah: Okay. Record both the owner and deadline as          │
│  unresolved for that task. [Line 77] [Line 38] Sarah: Let's summarize the action items.                         │
│                                                                                                                 │
│  Let's leave the owner unresolved for now and come back to it after we estimate the remaining work. [Line 57]   │
│  [Line 28] Sarah: The proposal also needs to be completed before the next project review. [Line 59] [Line 29]   │
│  Sarah: We haven't decided the exact date for the review yet. [Line 61] [Line 30] David: So the deadline for    │
│  the proposal is not known yet. [Line 63] [Line 31] Sarah: Correct. We'll determine the date later. [Line 65]   │
│  [Line 32] Sarah: There is another task we need to capture.                                                     │
│                                                                                                                 │
│  [Line 93] [Line 46] Sarah: The deadline for the project proposal also needs to be determined. [Line 95] [Line  │
│  47] Sarah: The remaining development work also needs to be estimated. [Line 97] [Line 48] Sarah: The person    │
│  responsible for the estimate has not been assigned. [Line 99] [Line 49] Sarah: The deadline for completing     │
│  the estimate has also not been assigned. [Line 101] [Line 50] Sarah: These unresolved items should be          │
│  clarified before we finalize the project plan.                                                                 │
│                                                                                                                 │
│      Please use only the information from the meeting context. Strictly avoid information outside               │
│      the retrieved context.                                                                                     │
│                                                                                                                 │
│                                                        

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meeting Summary Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Final MeetMind AI Meeting Report (using reviewed information only)                                             │
│                                                                                                                 │
│  1) Answers to the user question                                                                                │
│  - Which action items still need an owner or deadline?                                                          │
│    - Owner: The user-provided clarifications were applied — Sarah is the owner for all three action items       │
│  listed below (user clarification accepted; transcript did not contain these owners). Evidence that the         │
│  transcript left owners unresolved is included in the Unresolved Information section.                           │
│    - Deadline: All three action items still need deadlines. The transcript explicitly does not include any      │
│  dates; the user did not provide deadlines. Therefore deadlines remain UNRESOLVED for:                          │
│      1. Estimate the remaining development work — Deadline: UNRESOLVED                                          │
│      2. Determine the deadline for completing the project proposal (before the next project review) —           │
│  Deadline: UNRESOLVED                                                                                           │
│      3. Clarify unresolved items (assign owner/deadline for the estimate and determine the proposal deadline)   │
│  before finalizing the project plan — Deadline: UNRESOLVED                                                      │
│                                                                                                                 │
│  2) Relevant discussion points (supported by transcript)                                                        │
│  - Estimate the remaining development work — owner and deadline unresolved.                                     │
│    - Evidence (transcript quotes):                                                                              │
│      - "Someone needs to estimate the remaining development work."                                              │
│      - "We haven't assigned that task to anyone either."                                                        │
│      - "And we haven't set a deadline for the estimate."                                                        │
│      - "We can discuss the assignment after this meeting."                                                      │
│      - "Okay. Record both the owner and deadline as unresolved for that task."                                  │
│    - Transcript line references: [Line 67] [Line 33], [Line 69] [Line 34], [Line 71] [Line 35], [Line 73]       │
│  [Line 36], [Line 75] [Line 37], [Line 77] [Line 38], [Line 95] [Line 47], [Line 97] [Line 48], [Line 99]       │
│  [Line 49]                                                                                                      │
│                                                                                                                 │
│  - Deadline for the project proposal needs to be determined (deadline unknown).                                 │
│    - Evidence (transcript quotes):                                                                              │
│      - "The proposal also needs to be completed before 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create the final MeetMind AI meeting report.                                                             │
│                                                                                                                 │
│      User Question:What are the action items that still need an owner or deadline?                              │
│      User Clarifications: Owner clarification for Estimate the remaining development work: Sarah                │
│  Owner clarification for Determine the deadline for completing the project proposal (before the next project    │
│  review): Sarah                                                                                                 │
│  Owner clarification for Clarify unresolved items (estimate owner/deadline and proposal deadline) before        │
│  finalizing the project plan: Sarah                                                                             │
│                                                                                                                 │
│      use ONLY the reviewed information. DO NOT include the information rejected by the Review Agent.            │
│      Make sure to mark the unresolved information.                                                              │
│      Include evidences and transcript refererences for decisions and action items.                              │
│                                                                                                                 │
│                                                                                                                 │
│  Agent: Meeting Summary Specialist                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0bb56d20-4064-4893-9343-2b3eb7030be3                                                                       │
│  Final Output: Final MeetMind AI Meeting Report (using reviewed information only)                               │
│                                                                                                                 │
│  1) Answers to the user question                                                                                │
│  - Which action items still need an owner or deadline?                                                          │
│    - Owner: The user-provided clarifications were applied — Sarah is the owner for all three action items       │
│  listed below (user clarification accepted; transcript did not contain these owners). Evidence that the         │
│  transcript left owners unresolved is included in the Unresolved Information section.                           │
│    - Deadline: All three action items still need deadlines. The transcript explicitly does not include any      │
│  dates; the user did not provide deadlines. Therefore deadlines remain UNRESOLVED for:                          │
│      1. Estimate the remaining development work — Deadline: UNRESOLVED                                          │
│      2. Determine the deadline for completing the project proposal (before the next project review) —           │
│  Deadline: UNRESOLVED                                                                                           │
│      3. Clarify unresolved items (assign owner/deadline for the estimate and determine the proposal deadline)   │
│  before finalizing the project plan — Deadline: UNRESOLVED                                                      │
│                                                                                                                 │
│  2) Relevant discussion points (supported by transcript)                                                        │
│  - Estimate the remaining development work — owner and deadline unresolved.                                     │
│    - Evidence (transcript quotes):                                                                              │
│      - "Someone needs to estimate the remaining development work."                                              │
│      - "We haven't assigned that task to anyone either."                                                        │
│      - "And we haven't set a deadline for the estimate."                                                        │
│      - "We can discuss the assignment after this meeting."                                                      │
│      - "Okay. Record both the owner and deadline as unresolved for that task."                                  │
│    - Transcript line references: [Line 67] [Line 33], [Line 69] [Line 34], [Line 71] [Line 35], [Line 73]       │
│  [Line 36], [Line 75] [Line 37], [Line 77] [Line 38], [Line 95] [Line 47], [Line 97] [Line 48], [Line 99]       │
│  [Line 49]                                                                                                      │
│                                                                                                                 │
│  - Deadline for the project proposal needs to be determined (deadline unknown).                                 │
│    - Evidence (transcript quotes):                                                                              │
│      - "The proposal also needs to be completed before


MeetMind AI Final Report

----------------------------------------
Final MeetMind AI Meeting Report (using reviewed information only)

1) Answers to the user question
- Which action items still need an owner or deadline?
  - Owner: The user-provided clarifications were applied — Sarah is the owner for all three action items listed below (user clarification accepted; transcript did not contain these owners). Evidence that the transcript left owners unresolved is included in the Unresolved Information section.
  - Deadline: All three action items still need deadlines. The transcript explicitly does not include any dates; the user did not provide deadlines. Therefore deadlines remain UNRESOLVED for:
    1. Estimate the remaining development work — Deadline: UNRESOLVED
    2. Determine the deadline for completing the project proposal (before the next project review) — Deadline: UNRESOLVED
    3. Clarify unresolved items (assign owner/deadline for the estimate and determine the proposal de

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯